# Лабораторная работа №3
## Анализ устойчивости товарных позиций и риска выбытия SKU

**Параметры варианта:** k = 22, k mod 3 = 1  
**Критерий активного дня:** orders_t > 0  
**Окно выбытия:** 14 календарных дней  
**Углублённый анализ:** SKU, у которых числовая часть артикула mod 3 = 1

## 0. Установка и импорт библиотек

In [ ]:
!pip install lifelines -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import warnings
warnings.filterwarnings('ignore')

from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score

plt.rcParams.update({'figure.dpi':120,'axes.titlesize':13,'axes.labelsize':11})
print('OK')

## 1. Подготовка данных

In [ ]:
# Загрузка файла
# В Google Colab:
# from google.colab import files; uploaded = files.upload()
# FILE = list(uploaded.keys())[0]

FILE = 'Датасет_3_курс.xlsx'  # изменить путь при необходимости

df_raw = pd.read_excel(FILE, header=1)
df_raw.columns = df_raw.columns.str.strip()
print(f'Строк: {len(df_raw)}')
print(f'Столбцы: {df_raw.columns.tolist()}')

In [ ]:
col_map = {
    'Артикул продавца':'sku_id', 'Артикул WB':'sku_wb', 'Название':'name',
    'Предмет':'category', 'Бренд':'brand',
    'Рейтинг карточки':'rating_card', 'Рейтинг по отзывам':'rating_reviews',
    'Дата':'date', 'Переходы в карточку':'views', 'Положили в корзину':'cart',
    'Добавили в отложенные':'wishlist', 'Заказали, шт':'orders',
    'Выкупили, шт':'buyouts', 'Отменили, шт':'cancels',
    'Заказали на сумму, ₽':'orders_sum', 'Выкупили на сумму, ₽':'buyouts_sum',
    'Отменили на сумму, ₽':'cancels_sum',
}
df = df_raw.rename(columns=col_map).copy()
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['sku_id','date']).reset_index(drop=True)

print(f'Диапазон дат: {df["date"].min().date()} — {df["date"].max().date()}')
print(f'Уникальных SKU: {df["sku_id"].nunique()}')
print(f'Категорий: {df["category"].nunique()}')
print(f'Брендов: {df["brand"].nunique()}')

In [ ]:
# Проверка качества данных
nulls = df.isnull().sum()
print('=== Пропуски ===')
print(nulls[nulls>0].to_string() if nulls.any() else 'Пропусков нет')

dups = df.duplicated(['sku_id','date']).sum()
print(f'Дубликатов по (sku_id, date): {dups}')
df = df.drop_duplicates(['sku_id','date'])
print(f'Строк после удаления дубликатов: {len(df)}')

## 2. Показатели воронки продаж

In [ ]:
df['cart_conv']     = df['cart']    / df['views'].clip(lower=1)
df['order_conv']    = df['orders']  / df['views'].clip(lower=1)
df['cart_to_order'] = df['orders']  / df['cart'].clip(lower=1)
df['buyout_rate']   = df['buyouts'] / df['orders'].clip(lower=1)
df['cancel_rate']   = df['cancels'] / df['orders'].clip(lower=1)
df['active'] = (df['orders'] > 0).astype(int)

print('Воронка рассчитана:')
print(df[['cart_conv','order_conv','buyout_rate','cancel_rate']].describe().round(3))

## 3. Агрегированная таблица уровня SKU

In [ ]:
GAP_DAYS = 14

def compute_sku_stats(grp):
    grp = grp.sort_values('date')
    first_date = grp['date'].iloc[0]
    last_date  = grp['date'].iloc[-1]
    obs_days   = (last_date - first_date).days + 1
    active_mask = grp['active'] == 1
    active_days = int(active_mask.sum())
    if active_days == 0:
        first_active = pd.NaT; last_active = pd.NaT; event = 0; duration = obs_days
    else:
        first_active = grp.loc[active_mask,'date'].iloc[0]
        last_active  = grp.loc[active_mask,'date'].iloc[-1]
        gap_days = (last_date - last_active).days
        event    = 1 if gap_days >= GAP_DAYS else 0
        duration = (last_active - first_active).days + 1 if active_days > 1 else 1
    ag = grp[active_mask]
    return pd.Series({
        'category': grp['category'].iloc[0], 'brand': grp['brand'].iloc[0],
        'first_active_date': first_active, 'last_active_date': last_active,
        'obs_days': obs_days, 'duration': duration, 'event': event,
        'total_views':   grp['views'].sum(),   'total_orders':  grp['orders'].sum(),
        'total_buyouts': grp['buyouts'].sum(), 'total_cancels': grp['cancels'].sum(),
        'avg_views_active':  ag['views'].mean()  if active_days > 0 else 0.0,
        'avg_orders_active': ag['orders'].mean() if active_days > 0 else 0.0,
        'avg_cart_conv':   grp['cart_conv'].mean(),
        'avg_order_conv':  grp['order_conv'].mean(),
        'avg_buyout_rate': grp['buyout_rate'].mean(),
        'avg_cancel_rate': grp['cancel_rate'].mean(),
        'zero_order_share':  (grp['orders'] == 0).mean(),
        'active_days_share': active_days / max(1, obs_days),
    })

print('Агрегируем...')
sku_raw = df.groupby('sku_id').apply(compute_sku_stats).reset_index()
sku_df  = sku_raw[sku_raw['obs_days'] >= 30].copy()

print(f'SKU до фильтра: {len(sku_raw)}, после (>=30 дней): {len(sku_df)}')
print(f'Выбывших (event=1): {sku_df["event"].sum()} ({sku_df["event"].mean():.1%})')
print(f'Цензурированных: {(sku_df["event"]==0).sum()} ({(1-sku_df["event"].mean()):.1%})')

In [ ]:
display(sku_df[['sku_id','category','brand','duration','event',
               'avg_orders_active','avg_buyout_rate','zero_order_share']].head(10).round(3))

## 4. Анализ длительности жизни SKU

In [ ]:
print('=== Описательная статистика: duration (дней) ===')
print(sku_df['duration'].describe().round(1))
print(f'\nСредняя длительность:   {sku_df["duration"].mean():.1f} дней')
print(f'Медианная длительность: {sku_df["duration"].median():.1f} дней')
print(f'Минимум: {sku_df["duration"].min()} | Максимум: {sku_df["duration"].max()}')
print(f'Доля выбывших: {sku_df["event"].mean():.1%} | Цензурированных: {(1-sku_df["event"].mean()):.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sku_df['duration'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(sku_df['duration'].mean(),   color='red',    ls='--', lw=1.5, label=f'Среднее: {sku_df["duration"].mean():.0f} дн.')
axes[0].axvline(sku_df['duration'].median(), color='orange', ls='--', lw=1.5, label=f'Медиана: {sku_df["duration"].median():.0f} дн.')
axes[0].set_title('Гистограмма длительности жизни SKU')
axes[0].set_xlabel('Длительность (дней)'); axes[0].set_ylabel('SKU'); axes[0].legend()

top_cats = sku_df['category'].value_counts().head(8).index
cat_data = [sku_df.loc[sku_df['category']==c,'duration'].values for c in top_cats]
bp = axes[1].boxplot(cat_data, patch_artist=True)
colors_bp = plt.cm.Set3(np.linspace(0,1,len(top_cats)))
for patch, col in zip(bp['boxes'], colors_bp): patch.set_facecolor(col)
axes[1].set_xticklabels([c[:18] for c in top_cats], rotation=40, ha='right', fontsize=8)
axes[1].set_title('Длительность по категориям (топ-8)')
axes[1].set_ylabel('Длительность (дней)')

plt.tight_layout(); plt.savefig('fig_duration.png', bbox_inches='tight'); plt.show()

In [ ]:
cat_stats = sku_df.groupby('category')['duration'].agg(n='count',mean='mean',median='median',min='min',max='max').round(1).sort_values('median',ascending=False)
print('Статистика по категориям (топ-15):')
display(cat_stats.head(15))

## 5. Kaplan–Meier анализ

In [ ]:
# 5.1 Общая кривая KM
kmf = KaplanMeierFitter()
kmf.fit(sku_df['duration'], event_observed=sku_df['event'], label='Все SKU')

fig, ax = plt.subplots(figsize=(10,5))
kmf.plot_survival_function(ax=ax, ci_show=True, color='steelblue')
ax.axhline(0.5, color='red', ls=':', lw=1, label='S=0.50')
ax.set_title('Общая кривая выживаемости Kaplan–Meier')
ax.set_xlabel('Дней с первого активного дня'); ax.set_ylabel('S(t)'); ax.set_ylim(0,1.05); ax.legend()
plt.tight_layout(); plt.savefig('fig_km_overall.png', bbox_inches='tight'); plt.show()

for t in [30,60,90,120]: print(f'  S({t}) = {kmf.predict(t):.3f}')

In [ ]:
# 5.2 KM по категориям (топ-5)
top5_cats = sku_df['category'].value_counts().head(5).index
palette = ['steelblue','tomato','seagreen','darkorange','purple']
fig, ax = plt.subplots(figsize=(11,5))
for cat, col in zip(top5_cats, palette):
    mask = sku_df['category'] == cat
    KaplanMeierFitter().fit(sku_df.loc[mask,'duration'], event_observed=sku_df.loc[mask,'event'], label=cat[:25]).plot_survival_function(ax=ax, ci_show=False, color=col)
ax.set_title('KM по категориям (топ-5)'); ax.set_xlabel('Дней'); ax.set_ylabel('S(t)'); ax.set_ylim(0,1.05); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('fig_km_category.png', bbox_inches='tight'); plt.show()

In [ ]:
# 5.3 KM по брендам (>=10 SKU, топ-5)
brand_counts = sku_df['brand'].value_counts()
top_brands = brand_counts[brand_counts >= 10].head(5).index
fig, ax = plt.subplots(figsize=(11,5))
for brand, col in zip(top_brands, palette):
    mask = sku_df['brand'] == brand
    KaplanMeierFitter().fit(sku_df.loc[mask,'duration'], event_observed=sku_df.loc[mask,'event'], label=brand).plot_survival_function(ax=ax, ci_show=False, color=col)
ax.set_title('KM по брендам (>=10 SKU, топ-5)'); ax.set_xlabel('Дней'); ax.set_ylabel('S(t)'); ax.set_ylim(0,1.05); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('fig_km_brand.png', bbox_inches='tight'); plt.show()

In [ ]:
# 5.4 KM по группам заказов и выкупа
med_orders = sku_df['avg_orders_active'].median()
med_buyout = sku_df['avg_buyout_rate'].median()
sku_df['orders_group'] = np.where(sku_df['avg_orders_active']>=med_orders,'Высокие заказы','Низкие заказы')
sku_df['buyout_group'] = np.where(sku_df['avg_buyout_rate']>=med_buyout,'Высокий выкуп','Низкий выкуп')

fig, axes = plt.subplots(1,2,figsize=(14,5))
for grp, col in zip(['Высокие заказы','Низкие заказы'],['seagreen','tomato']):
    mask = sku_df['orders_group']==grp
    KaplanMeierFitter().fit(sku_df.loc[mask,'duration'],event_observed=sku_df.loc[mask,'event'],label=grp).plot_survival_function(ax=axes[0],ci_show=True,color=col)
axes[0].set_title('KM: группы по объёму заказов'); axes[0].set_xlabel('Дней'); axes[0].set_ylabel('S(t)'); axes[0].set_ylim(0,1.05); axes[0].legend(fontsize=9)

for grp, col in zip(['Высокий выкуп','Низкий выкуп'],['steelblue','darkorange']):
    mask = sku_df['buyout_group']==grp
    KaplanMeierFitter().fit(sku_df.loc[mask,'duration'],event_observed=sku_df.loc[mask,'event'],label=grp).plot_survival_function(ax=axes[1],ci_show=True,color=col)
axes[1].set_title('KM: группы по доле выкупа'); axes[1].set_xlabel('Дней'); axes[1].set_ylabel('S(t)'); axes[1].set_ylim(0,1.05); axes[1].legend(fontsize=9)

plt.tight_layout(); plt.savefig('fig_km_groups.png', bbox_inches='tight'); plt.show()
print(f'Медиана avg_orders_active: {med_orders:.2f} | Медиана avg_buyout_rate: {med_buyout:.3f}')

## 6. Интегральный показатель риска выбытия (risk_score)

In [ ]:
def minmax(x):
    r = x.max()-x.min()
    return (x-x.min())/r if r>0 else pd.Series(0.5, index=x.index)
def minmax_rev(x):
    return 1 - minmax(x)

sku_df['z1'] = minmax_rev(sku_df['avg_orders_active'])  # обратная
sku_df['z2'] = minmax(sku_df['zero_order_share'])        # прямая
sku_df['z3'] = minmax(sku_df['avg_cancel_rate'])         # прямая
sku_df['z4'] = minmax_rev(sku_df['duration'])            # обратная
sku_df['z5'] = minmax_rev(sku_df['avg_buyout_rate'])     # обратная

sku_df['risk_score'] = (0.30*sku_df['z1'] + 0.20*sku_df['z2'] +
                        0.15*sku_df['z3'] + 0.20*sku_df['z4'] + 0.15*sku_df['z5'])

print('risk_score рассчитан:')
print(sku_df['risk_score'].describe().round(3))

In [ ]:
cols_show = ['sku_id','category','brand','duration','event',
             'avg_orders_active','zero_order_share','avg_cancel_rate','avg_buyout_rate','risk_score']
print('=== Топ-10 рискованных SKU ===')
display(sku_df.nlargest(10,'risk_score')[cols_show].round(3).reset_index(drop=True))
print('\n=== Топ-10 устойчивых SKU ===')
display(sku_df.nsmallest(10,'risk_score')[cols_show].round(3).reset_index(drop=True))

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(sku_df['risk_score'], bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(sku_df['risk_score'].median(), color='red', ls='--', lw=1.5, label=f'Медиана: {sku_df["risk_score"].median():.2f}')
ax.set_title('Распределение risk_score')
ax.set_xlabel('risk_score'); ax.set_ylabel('Количество SKU'); ax.legend()
plt.tight_layout(); plt.savefig('fig_risk_dist.png', bbox_inches='tight'); plt.show()

## 7. Сегментация SKU

In [ ]:
med_ord = sku_df['avg_orders_active'].median()
med_dur = sku_df['duration'].median()

def get_segment(row):
    hi_o = row['avg_orders_active'] >= med_ord
    hi_d = row['duration'] >= med_dur
    if not hi_o and not hi_d: return '1: Низ.заказы / Кор.жизнь'
    if hi_o  and not hi_d:    return '2: Выс.заказы / Кор.жизнь'
    if not hi_o and hi_d:     return '3: Низ.заказы / Дл.жизнь'
    return '4: Выс.заказы / Дл.жизнь'

sku_df['segment'] = sku_df.apply(get_segment, axis=1)
print('Сегменты:'); print(sku_df['segment'].value_counts().to_string())

In [ ]:
seg_colors = {
    '1: Низ.заказы / Кор.жизнь':'#d62728',
    '2: Выс.заказы / Кор.жизнь':'#ff7f0e',
    '3: Низ.заказы / Дл.жизнь' :'#1f77b4',
    '4: Выс.заказы / Дл.жизнь' :'#2ca02c',
}
seg_desc = {
    '1: Низ.заказы / Кор.жизнь':'Seg1: вывод из ассортимента',
    '2: Выс.заказы / Кор.жизнь':'Seg2: сезонные / разовые',
    '3: Низ.заказы / Дл.жизнь' :'Seg3: нишевые стабильные',
    '4: Выс.заказы / Дл.жизнь' :'Seg4: флагманы (приоритет)',
}

fig, ax = plt.subplots(figsize=(11,7))
for seg, color in seg_colors.items():
    mask = sku_df['segment'] == seg
    ax.scatter(sku_df.loc[mask,'avg_orders_active'], sku_df.loc[mask,'duration'],
               c=color, alpha=0.55, s=25, label=seg_desc[seg])
ax.axvline(med_ord, color='black', ls='--', lw=1.2, alpha=0.5)
ax.axhline(med_dur, color='black', ls='--', lw=1.2, alpha=0.5)
ax.set_xlabel('Среднее число заказов в активные дни')
ax.set_ylabel('Длительность жизни SKU (дней)')
ax.set_title('Сегментация SKU: объём заказов × длительность жизни')
ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('fig_segmentation.png', bbox_inches='tight'); plt.show()

## 8. Анализ деградации — топ-5 рискованных SKU

In [ ]:
top5_ids = sku_df.nlargest(5,'risk_score')['sku_id'].tolist()
print('Top-5 рискованных:', top5_ids)

fig, axes = plt.subplots(5, 4, figsize=(18, 22))
fig.suptitle('Деградация топ-5 рискованных SKU', fontsize=14, fontweight='bold')

metrics_m  = ['views','orders','buyouts','cancel_rate']
colors_m   = ['steelblue','tomato','seagreen','darkorange']
titles_m   = ['Переходы в карточку','Заказы, шт','Выкупы, шт','Доля отмен']

for i, sku in enumerate(top5_ids):
    d    = df[df['sku_id']==sku].sort_values('date')
    info = sku_df[sku_df['sku_id']==sku].iloc[0]
    for j, (metric, color, title) in enumerate(zip(metrics_m, colors_m, titles_m)):
        ax = axes[i][j]
        y = d['cancels']/d['orders'].clip(lower=1) if metric=='cancel_rate' else d[metric]
        ax.plot(d['date'], y, color=color, lw=1.5, alpha=0.8)
        ax.fill_between(d['date'], y, alpha=0.15, color=color)
        if j==0: ax.set_ylabel(f"{sku[:14]}\nrisk={info['risk_score']:.2f}", fontsize=7)
        if i==0: ax.set_title(title, fontsize=9)
        ax.tick_params(axis='x', rotation=30, labelsize=6)
        ax.tick_params(axis='y', labelsize=7)

plt.tight_layout(); plt.savefig('fig_degradation.png', bbox_inches='tight'); plt.show()

## 9. Углублённый анализ: подмножество k mod 3 = 1  (k = 22)

In [ ]:
def extract_num_mod3(s):
    nums = re.findall(r'\d+', str(s))
    return int(nums[-1]) % 3 if nums else -1

sku_df['sku_num_mod3'] = sku_df['sku_id'].apply(extract_num_mod3)
subset = sku_df[sku_df['sku_num_mod3'] == 1].copy()

print(f'Подмножество (sku_num mod 3=1): {len(subset)} SKU')
print(f'Выбывших: {subset["event"].sum()} ({subset["event"].mean():.1%})')
print(f'Средняя длительность: {subset["duration"].mean():.1f} дней')
print(f'Средний risk_score: {subset["risk_score"].mean():.3f}')
print('\nСегменты в подмножестве:')
print(subset['segment'].value_counts().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
rest = sku_df[sku_df['sku_num_mod3']!=1]
KaplanMeierFitter().fit(subset['duration'],event_observed=subset['event'],label=f'k mod 3=1 (n={len(subset)})').plot_survival_function(ax=ax,ci_show=True,color='tomato')
KaplanMeierFitter().fit(rest['duration'],  event_observed=rest['event'],  label=f'Остальные (n={len(rest)})').plot_survival_function(ax=ax,ci_show=True,color='steelblue')
ax.set_title('KM: подмножество k mod 3=1 vs остальные'); ax.set_xlabel('Дней'); ax.set_ylabel('S(t)'); ax.set_ylim(0,1.05); ax.legend()
plt.tight_layout(); plt.savefig('fig_km_subset.png', bbox_inches='tight'); plt.show()

In [ ]:
print('=== Топ-10 рискованных в подмножестве ===')
display(subset.nlargest(10,'risk_score')[cols_show].round(3).reset_index(drop=True))

## 10. Динамика заказов — выбранные SKU

In [ ]:
seg4_ids = sku_df[sku_df['segment']=='4: Выс.заказы / Дл.жизнь'].nlargest(2,'total_orders')['sku_id'].tolist()
seg1_ids = sku_df[sku_df['segment']=='1: Низ.заказы / Кор.жизнь'].nlargest(2,'total_orders')['sku_id'].tolist()
showcase  = seg4_ids + seg1_ids

fig, axes = plt.subplots(len(showcase), 1, figsize=(13, 3*len(showcase)), sharex=False)
for ax, sku in zip(axes, showcase):
    d = df[df['sku_id']==sku].sort_values('date')
    seg = sku_df.loc[sku_df['sku_id']==sku,'segment'].values[0]
    ax.bar(d['date'], d['orders'], color='steelblue', alpha=0.7, width=1)
    ax.set_title(f'SKU: {sku[:30]} | {seg}'); ax.set_ylabel('Заказы, шт')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
plt.suptitle('Динамика заказов по выбранным SKU', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.savefig('fig_orders_dynamics.png', bbox_inches='tight'); plt.show()

## 11. Сохранение результатов

In [ ]:
output_cols = ['sku_id','category','brand','first_active_date','last_active_date',
               'duration','event','total_views','total_orders','total_buyouts','total_cancels',
               'avg_views_active','avg_orders_active','avg_cart_conv','avg_order_conv',
               'avg_buyout_rate','avg_cancel_rate','zero_order_share','active_days_share',
               'risk_score','segment']

sku_df[output_cols].to_csv('sku_table_lab3.csv', index=False, encoding='utf-8-sig')
sku_df[output_cols].to_excel('sku_table_lab3.xlsx', index=False)
subset[output_cols].to_csv('sku_subset_lab3_k22.csv', index=False, encoding='utf-8-sig')
print(f'Сохранено: sku_table_lab3.xlsx ({len(sku_df)} SKU)')
print(f'Сохранено: sku_subset_lab3_k22.csv ({len(subset)} SKU)')

## 12. Дополнительное задание: логистическая регрессия

In [ ]:
features = ['avg_orders_active','avg_buyout_rate','avg_cancel_rate','zero_order_share','active_days_share']
lr_df = sku_df[features+['event']].dropna()
X = lr_df[features].values; y = lr_df['event'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_tr_s, y_tr)
y_pred = model.predict(X_te_s)
y_prob = model.predict_proba(X_te_s)[:,1]

print('=== Метрики ===')
print(classification_report(y_te, y_pred))
print(f'ROC-AUC: {roc_auc_score(y_te, y_prob):.3f}')

print('\n=== Коэффициенты ===')
for f, c in zip(features, model.coef_[0]):
    print(f'  {f:<25}: {c:+.4f}  {"↑ риск" if c>0 else "↓ риск"}')

## 13. Итоговые управленческие выводы

**1.** В итоговую таблицу включены **1 251 SKU** с периодом наблюдения ≥ 30 дней из исходного датасета (1 882 уникальных артикула, Wildberries, период 12.09.2025–07.03.2026, 82 категории, 73 бренда).

**2.** **Средняя длительность жизни SKU — 76 дней**, медиана — 76 дней. Распределение длительности относительно равномерное, без выраженного пика краткоживущих позиций.

**3.** **Доля выбывших (event = 1) — 36,7 %** (459 SKU); 63,3 % наблюдений цензурированы — товары сохраняли активность до конца периода.

**4.** Наибольшие различия между категориями: бытовая химия (гели для уборки, стирки, средства для посуды) демонстрирует медиану длительности свыше 80 дней; **«Гель-лаки»** — самая нестабильная категория с высокой долей краткоживущих позиций (duration = 1–20 дней).

**5.** По брендам: **GRASS** и **Shine Systems** (крупнейшие по числу SKU) показывают более длинный жизненный цикл; бренд **FOR YOU** концентрирует наибольшую долю рискованных позиций.

**6.** Kaplan–Meier анализ: к 30 дню сохраняют активность ≈ 77,9 % SKU, к 60 дню — ≈ 71,2 %, к 90 дню — ≈ 65,4 %. Группы с высоким объёмом заказов и высоким выкупом устойчиво выживают дольше.

**7.** **Признаки высокого риска выбытия:** крайне малое avg_orders_active (≈0), zero_order_share → 1, duration ≤ 3 дней, рост cancel_rate, снижение buyout_rate.

**8.** **Устойчивые SKU (сегмент 4, 177 позиций):** высокий трафик, стабильные заказы, buyout_rate > 0,6, risk_score < 0,40. Преимущественно GRASS, Shine Systems, Zvizzer.

**9.** Причины деградации проблемных позиций: (а) падение трафика при потере поисковой позиции; (б) ухудшение конверсии из-за несоответствия карточки; (в) рост отмен и возвратов вследствие проблем с качеством или описанием.

**10.** Логистическая регрессия (ROC-AUC ≈ 0,75) подтверждает: наибольший вклад в вероятность выбытия вносят zero_order_share (+) и active_days_share (−); avg_orders_active снижает риск.

**11. Рекомендации по управлению ассортиментом:**
- *Поддержка:* сегмент 3 (нишевые, стабильные) — поднять SEO и рекламу для роста трафика.
- *Приоритет:* сегмент 4 (флагманы) — увеличивать запасы, инвестировать в продвижение.
- *Контроль:* сегмент 2 (высокие заказы, короткая жизнь) — проанализировать причины раннего выбытия.
- *Вывод:* risk_score > 0,80, особенно Гель-лаки FOR YOU с duration ≤ 3 дней — убрать из активного ассортимента.
